In [8]:
import sqlite3
import pandas as pd
import numpy as np

conn = sqlite3.connect('espn_cfb_2026.db')

pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)

,name
0,Teams
1,Games
2,Box_Scores
3,Player_Box_Scores
4,Raw_Boxscores
5,Scoreboard_Fetches
6,Player_Boxscore_Fetches
7,Team_Defensive_Stats
8,Team_Defense_Opponent_Stats
9,Raw_Plays


In [9]:
load_games = """
SELECT game_id, week, home_team_id, away_team_id, home_score, away_score, neutral_site
FROM Games
WHERE status = 'Final'
"""

games = pd.read_sql_query(load_games, conn)

print(games.shape)
display(games.head())
display(games.dtypes)

(752, 7)


,game_id,week,home_team_id,away_team_id,home_score,away_score,neutral_site
0,401864494,1,30,23,42,26,0
1,401856766,1,2628,153,10,15,1
2,401858202,1,258,152,34,8,0
3,401864577,1,2449,55,33,7,0
4,401866408,1,2199,16,28,17,0


game_id           str
week            int64
home_team_id      str
away_team_id      str
home_score      int64
away_score      int64
neutral_site    int64
dtype: object

In [10]:
load_teams = """
SELECT team_id, short_name, conference
FROM Teams
"""

teams = pd.read_sql_query(load_teams, conn)

teams["conference"].value_counts(dropna=False)

conference
NaN                          102
Big Ten Conference            18
Atlantic Coast Conference     17
Big 12 Conference             16
Southeastern Conference       16
American Conference           14
Sun Belt Conference           14
Mid-American Conference       13
Conference USA                10
Mountain West Conference      10
Pac-12 Conference              8
FBS Independents               2
Name: count, dtype: int64

In [11]:
fcs_ids = set(teams[teams["conference"].isna()]["team_id"])

games["home_team_id"] = games["home_team_id"].mask(games["home_team_id"].isin(fcs_ids), "FCS")
games["away_team_id"] = games["away_team_id"].mask(games["away_team_id"].isin(fcs_ids), "FCS")

new_row = pd.DataFrame([{"team_id": "FCS", "short_name": "FCS", "conference": "FCS"}])
teams = teams if "FCS" in teams["team_id"].values else pd.concat([teams, new_row], ignore_index=True)

teams.shape

(241, 3)

In [12]:
home_fcs = games["home_team_id"].eq("FCS")
away_fcs = games["away_team_id"].eq("FCS")

print("Home FCS games:", home_fcs.sum())
print("Away FCS games:", away_fcs.sum())
print("Games vs. FCS (either side):", (home_fcs | away_fcs).sum())
print("FCS vs. FCS (both sides):", (home_fcs & away_fcs).sum())

Home FCS games: 0
Away FCS games: 123
Games vs. FCS (either side): 123
FCS vs. FCS (both sides): 0


In [13]:
home_view = games.rename(columns={
	"home_team_id": "team_id",
	"away_team_id": "opp_id",
	"home_score": "points_for",
	"away_score": "points_against",
})

away_view = games.rename(columns={
	"away_team_id": "team_id",
	"home_team_id": "opp_id",
	"away_score": "points_for",
	"home_score": "points_against",
})

home_view["location"] = np.where(home_view["neutral_site"] == 1, "neutral", "home")
away_view["location"] = np.where(away_view["neutral_site"] == 1, "neutral", "away")

team_games = pd.concat([home_view, away_view], ignore_index=True)
team_games = team_games.drop(columns=["neutral_site"])

team_games["margin"] = team_games["points_for"] - team_games["points_against"]
team_games["win"] = team_games["margin"].gt(0).astype(int)

In [14]:
display(len(team_games))
display(team_games["game_id"].value_counts().unique())
display(team_games["margin"].sum())
display((team_games["margin"] == 0).sum())
display(team_games["location"].value_counts())

1504

array([2])

np.int64(0)

np.int64(0)

location
home       742
away       742
neutral     20
Name: count, dtype: int64